In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets

# Loading Data, Processing and Splitting Data
#### Dataset contains 87,000 images with 29 test data to support testing with new data

#### Image are 200x200 pixels

#### Dataset is categorized into 29 classes all alphabets, delete, space and nothing.

In [2]:
# Define image transforms (normalize)
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
    ]
)

In [3]:
# Point to the root folder on your PC
trainset_path = r"C:\Users\25471\Desktop\Jupyter Notebook\archive (3)\asl_alphabet_train\asl_alphabet_train"  #Change to your path
train_dataset = datasets.ImageFolder(root=trainset_path, transform=transform)


testset_path = r"C:\Users\25471\Desktop\Jupyter Notebook\archive (3)\asl_alphabet_test"
test_data = datasets.ImageFolder(root=testset_path, transform=transform)

In [4]:
# split the training dataset into train set and validation set (80%, 20%)
train_size = int(0.8 * len(train_dataset))
val_size = int(0.2 * len(train_dataset))


#splitting the train dataset into a training set and validation set
train_data, val_data = random_split(
    train_dataset, [train_size, val_size]
)

# create DataLoaders for batching and shuffling
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0) #change number of workers to 0 to avoid jupter notebook freezes
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False, num_workers=0) # used one batch size for test_loader since testing dataset is only 29 images, one of each class

In [5]:
alphabets = ('A', 'B', 'C','D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'Del', 'Space', 'Nothing') 

In [6]:
len(alphabets)

29

# Defining the Neural Net Architecture
### CNN consists of:

#### 3 convolutional layers

#### 2 maxpool

#### A dense layer

#### A flattening layer

#### Uses Relu transformation after every linear tranformation

#### And finally a softmax activation on the output layer

In [7]:
class ASLNet(nn.Module):
    def __init__(self):
        super(ASLNet, self).__init__()
        
        # Three Convolutional Layers with maxpool between the first and  second convolution layer
        # Input: 3 channels (RGB) -> Output: 32 channels
        self.conv1 = nn.Conv2d(3, 32, 3, padding = 1)
        
        # Input: 32 channels -> Output: 64 channels
        self.conv2 = nn.Conv2d(32, 64, 3, padding = 1)
        
        # Input: 64 channels -> Output: 128 channels
        self.conv3 = nn.Conv2d(64, 128, 3, padding = 1)

        # Pooling layer with a kernel size of 2*2 and stride of 2
        self.pool = nn.MaxPool2d(2,2) 
        
        # Dense (Linear) Layer
        # Math: 200x200 -> Pool 1 (100x100) -> Pool 2 (50x50)
        # Flattened features = 128 channels * 50 * 50 pixels = 320,000
        self.fc1 = nn.Linear(128 * 50 * 50, 512)
        self.out = nn.Linear(512, 29) # Output Layer configured for exactly 29 classes

    def forward(self, x):
        # Conv 1 -> ReLU -> MaxPool (Dimensions: 200x200 -> 100x100)
        x = self.pool(F.relu(self.conv1(x)))
        
        # Conv 2 -> ReLU -> MaxPool (Dimensions: 100x100 -> 50x50)
        x = self.pool(F.relu(self.conv2(x)))
        
        # Conv 3 -> ReLU (Dimensions stay 50x50)
        x = F.relu(self.conv3(x))
        
        # 5. Flattening Layer
        x = x.view(x.size(0), -1) 
        
        # Dense Layer -> ReLU
        x = F.relu(self.fc1(x))
        
        # 6. Output Layer -> Softmax
        x = F.softmax(self.out(x), dim=1)
        return x

In [8]:
ASLClassifier = ASLNet()

# Backward Propagation

#### Calculating loss, adjusting gradients and updating weights to improve model performance

#### Testing project with ADAM and SGD wit momentum to evaluate performance 

In [9]:
criterion = nn.CrossEntropyLoss() #sparse Cross Entropy Loss
optimizer = optim.SGD(ASLClassifier.parameters(), lr=0.001, momentum=0.9)#used SGD with momentum

In [10]:
# Training Loop
from datetime import datetime
elsapsed = None #total amount of time taken in training

def run_training_loop(epochs):
    start = datetime.now() #take timestamp immediately after the training loop starts
    batch_metrics = []
    epoch_metrics = []

    for epoch in range(epochs):
        ASLClassifier.train()
        running_loss = 0.0

        for i, data in enumerate(train_loader, 0):
            inputs, labels = data

            optimizer.zero_grad()

            outputs = ASLClassifier(inputs)

            loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                print(f"Non-finite loss detected at epoch {epoch + 1}, batch {i + 1}")
                print(f"Loss: {loss.item()}")
                print(f"Output range: {outputs.min().item()} to {outputs.max().item()}")
                raise RuntimeError("Non-finite loss detected")

            loss.backward()
            optimizer.step()

            loss_value = loss.item()
            running_loss += loss_value

            batch_metrics.append({
                "epoch": epoch + 1,
                "batch": i + 1,
                "loss": loss_value
            })

        training_loss = running_loss / len(train_loader)

        ASLClassifier.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = ASLClassifier(inputs)
                loss = criterion(outputs, labels)
                validation_loss += loss.item()

        validation_loss /= len(val_loader)

        epoch_metrics.append({
            "epoch": epoch + 1,
            "training_loss": training_loss,
            "validation_loss": validation_loss
        })
        elapsed = datetime.now() - start

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Training Loss: {training_loss:.4f} | "
            f"Validation Loss: {validation_loss:.4f} | "
            f"TimeElapsed: {elapsed}"
        )
    print("Finished Training")


In [11]:
run_training_loop(5)

Epoch 1/5 | Training Loss: 3.3668 | Validation Loss: 3.3660 | TimeElapsed: 2:23:54.243758
Epoch 2/5 | Training Loss: 3.3597 | Validation Loss: 3.3303 | TimeElapsed: 4:53:30.726361
Epoch 3/5 | Training Loss: 3.2391 | Validation Loss: 3.1701 | TimeElapsed: 7:11:22.431494
Epoch 4/5 | Training Loss: 3.1450 | Validation Loss: 3.1252 | TimeElapsed: 9:28:59.127440
Epoch 5/5 | Training Loss: 3.1014 | Validation Loss: 3.0920 | TimeElapsed: 11:35:12.748808
Finished Training


In [14]:
#calculating model accuracy using accuracy score

correct = 0
total = 0

incorrect_preds = {
    "image": [],
    "label": [],
    "prediction": []
}

ASLClassifier.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = ASLClassifier(images)
        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for i, prediction in enumerate(predicted):
            if prediction != labels[i]:
                incorrect_preds["image"].append(images[i])
                incorrect_preds["label"].append(labels[i].item())
                incorrect_preds["prediction"].append(prediction.item())

accuracy = correct / total * 100

print(f"Accuracy of the network on the 29 test images: {accuracy:.2f}%")

Accuracy of the network on the 29 test images: 17.86%


In [15]:
import matplotlib.pyplot as plt


def display_image_from_tensor(tensor, l, p):
    fig, ax = plt.subplots()
    mean = std =  torch.tensor([0.5, 0.5, 0.5]).view(3,1,1) # De-normalizing

    img_np = tensor.cpu().detach() * std + mean
    img_np = img_np.numpy().transpose(1,2,0)
    img_np = img_np.clip(0, 1)

    ax.imshow(img_np, interpolation="bilinear")
    print(f"Label: {classes[l]}, Predicted: {classes[p]}")
    
    plt.axis('off')
    plt.show()
    return